# Data Preparation

## What this notebook does
1. Loads the VAE latent vectors (β = 3×10⁻⁵) produced by the training pipeline  
2. Loads and preprocesses PPMI clinical assessment tables  
3. Computes UPDRS subscores and total scores  
4. Merges clinical data with latent vectors on `PATNO + EVENT_ID`  
5. Applies a **patient-stratified** train/val split (fixes data leakage in baseline)  
6. Fits the SBR PCA **once** on the training set and saves it for reuse  
7. Saves the final train and validation files  

## Improvements over baseline (Mahmoud's 5.0)
- Patient-stratified split: all visits of one patient stay in one split  
- Clinical scores added: UPDRS I–IV, MoCA, disease duration, REM sleep  
- SBR PCA fitted once and saved — not recomputed in every notebook  
- Explicit coverage report: how many rows have each clinical variable  


## 1. Imports and Configuration

In [2]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
import glob
warnings.filterwarnings('ignore')

# Paths
# Latent vectors (VAE output — beta = 3e-5)
LATENT_FILE = '../../data/baseline/final_train_combined_vae_data.csv'

# Raw PPMI clinical tables
CLINICAL_DIR = '../../data/raw/ppmi_clinical'

UPDRS_I_FILE   = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_I__Patient_Questionnaire-Archived_25Jun2026.csv')
UPDRS_II_FILE  = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_II__Patient_Questionnaire-Archived_25Jun2026.csv')
UPDRS_III_FILE = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_III-Archived_25Jun2026.csv')
UPDRS_IV_FILE  = os.path.join(CLINICAL_DIR, 'MDS_UPDRS_Part_IV-Archived_25Jun2026.csv')
MOCA_FILE      = os.path.join(CLINICAL_DIR, 'Montreal_Cognitive_Assessment__MoCA_-Archived_25Jun2026.csv')
PD_FEAT_FILE   = os.path.join(CLINICAL_DIR, 'PD_Features-Archived_25Jun2026.csv')
REM_FILE       = os.path.join(CLINICAL_DIR, 'REM_Sleep_Disorder_Questionnaire-Archived_25Jun2026.csv')

# Output paths
TRAIN_OUT  = '../../data/processed/clinical_merged/train.csv'
VAL_OUT    = '../../data/processed/clinical_merged/val.csv'
SCALER_OUT = '../../results/models/scaler_sbr.pkl'
PCA_OUT    = '../../results/models/pca_sbr.pkl'

# Parameters
TRAIN_RATIO        = 0.8
RANDOM_STATE       = 42
VARIANCE_THRESHOLD = 0.1   # latent dims with std below this treated as collapsed
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]
N_SBR_PCS   = 3
PATIENT_COL = 'PATNO'
LABEL_COL   = 'label'
JOIN_KEY    = ['PATNO', 'EVENT_ID']

# Create output directories
os.makedirs('../../data/processed/clinical_merged', exist_ok=True)
os.makedirs('../../results/models', exist_ok=True)

print("Configuration loaded.")
print(f"  Train ratio:        {TRAIN_RATIO}")
print(f"  Random state:       {RANDOM_STATE}")
print(f"  SBR columns:        {len(SBR_COLS)}")
print(f"  SBR PCs to extract: {N_SBR_PCS}")


# Check if the clinical data files exist

tables = {
    "MDS_UPDRS_Part_I":              UPDRS_I_FILE,
    "MDS_UPDRS_Part_II":             UPDRS_II_FILE,
    "MDS_UPDRS_Part_III":            UPDRS_III_FILE,
    "MDS_UPDRS_Part_IV":             UPDRS_IV_FILE,
    "Montreal_Cognitive_Assessment": MOCA_FILE,
    "PD_Features":                   PD_FEAT_FILE,
    "REM_Sleep":                     REM_FILE,
}

print("-" * 65)     
print("Checking for clinical data files...")
for name, pattern in tables.items():
    matches = glob.glob(pattern)
    if matches:
        df = pd.read_csv(matches[0], nrows=2)
        print(f" File:    {matches[0]}")
    else:
        print(f"✗ {name} — not found")
        print()

Configuration loaded.
  Train ratio:        0.8
  Random state:       42
  SBR columns:        6
  SBR PCs to extract: 3
-----------------------------------------------------------------
Checking for clinical data files...
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_I__Patient_Questionnaire-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_II__Patient_Questionnaire-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_III-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/MDS_UPDRS_Part_IV-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/Montreal_Cognitive_Assessment__MoCA_-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/PD_Features-Archived_25Jun2026.csv
 File:    ../../data/raw/ppmi_clinical/REM_Sleep_Disorder_Questionnaire-Archived_25Jun2026.csv


## 2. Load Latent Vector

In [3]:
# Load the latent vectors from the baseline(Mahmoud's) VAE model
df_latents = pd.read_csv(LATENT_FILE)

print(f"Shape:           {df_latents.shape}")
print(f"Unique patients: {df_latents[PATIENT_COL].nunique()}")
print(f"Total rows:      {len(df_latents)}")
print(f"\nLabel counts:")
print(df_latents[LABEL_COL].value_counts().to_string())
print(f"\nEVENT_ID distribution:")
print(df_latents['EVENT_ID'].value_counts().to_string())
print(f"\nRows per patient (mean): {len(df_latents) / df_latents[PATIENT_COL].nunique():.2f}")

Shape:           (2373, 303)
Unique patients: 1437
Total rows:      2373

Label counts:
label
PD         2030
Control     233
SWEDD       110

EVENT_ID distribution:
EVENT_ID
SC     1228
V06     392
V04     385
V10     256
U01      41
ST       32
V02      23
V05      10
U02       6

Rows per patient (mean): 1.65


## 3. Load and Preprocess Clinical Tables

### 3.1 MDS-UPDRS Part I — Non-motor symptoms

In [4]:
df_updrs1 = pd.read_csv(UPDRS_I_FILE)

# UPDRS Part I items
UPDRS1_ITEMS = ['NP1SLPN', 'NP1SLPD', 'NP1PAIN', 'NP1URIN',
                'NP1CNST', 'NP1LTHD', 'NP1FATG']

# Compute total score (sum of items, require at least 5 of 7 non-null)
df_updrs1['UPDRS1_TOTAL'] = df_updrs1[UPDRS1_ITEMS].sum(axis=1, min_count=5)
# print(df_updrs1['UPDRS1_TOTAL'].head(2))

df_updrs1 = df_updrs1[JOIN_KEY + ['UPDRS1_TOTAL']].dropna(subset=['UPDRS1_TOTAL'])

print(f"UPDRS-I loaded:  {df_updrs1.shape[0]} rows, {df_updrs1[PATIENT_COL].nunique()} patients")
print(f"Score range:     {df_updrs1['UPDRS1_TOTAL'].min():.0f} – {df_updrs1['UPDRS1_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_updrs1['UPDRS1_TOTAL'].mean():.1f} ± {df_updrs1['UPDRS1_TOTAL'].std():.1f}")

UPDRS-I loaded:  13905 rows, 2136 patients
Score range:     0 – 25
Mean ± std:      5.2 ± 4.0


### 3.2 MDS-UPDRS Part II - Motor daily living


In [16]:
df_updrs2 = pd.read_csv(UPDRS_II_FILE)

UPDRS2_ITEMS = ['NP2SPCH', 'NP2SALV', 'NP2SWAL', 'NP2EAT', 'NP2DRES',
                'NP2HYGN', 'NP2HWRT', 'NP2HOBB', 'NP2TURN', 'NP2TRMR',
                'NP2RISE', 'NP2WALK', 'NP2FREZ']

df_updrs2['UPDRS2_TOTAL'] = df_updrs2[UPDRS2_ITEMS].sum(axis=1, min_count=10)

df_updrs2 = df_updrs2[JOIN_KEY + ['UPDRS2_TOTAL']].dropna(subset=['UPDRS2_TOTAL'])


print(f"UPDRS-II loaded: {df_updrs2.shape[0]} rows, {df_updrs2[PATIENT_COL].nunique()} patients")
print(f"Score range:     {df_updrs2['UPDRS2_TOTAL'].min():.0f} – {df_updrs2['UPDRS2_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_updrs2['UPDRS2_TOTAL'].mean():.1f} ± {df_updrs2['UPDRS2_TOTAL'].std():.1f}")

UPDRS-II loaded: 13904 rows, 2136 patients
Score range:     0 – 48
Mean ± std:      5.9 ± 6.4


### 3.3 MDS-UPDRS Part III - Motor examination (most important)

In [35]:
df_updrs3 = pd.read_csv(UPDRS_III_FILE)

UPDRS3_ITEMS = [
    'NP3SPCH', 'NP3FACXP', 'NP3RIGN', 'NP3RIGRU', 'NP3RIGLU',
    'PN3RIGRL', 'NP3RIGLL', 'NP3FTAPR', 'NP3FTAPL', 'NP3HMOVR',
    'NP3HMOVL', 'NP3PRSPR', 'NP3PRSPL', 'NP3TTAPR', 'NP3TTAPL',
    'NP3LGAGR', 'NP3LGAGL', 'NP3RISNG', 'NP3GAIT', 'NP3FRZGT',
    'NP3PSTBL', 'NP3POSTR', 'NP3BRADY', 'NP3PTRMR', 'NP3PTRML',
    'NP3KTRMR', 'NP3KTRML', 'NP3RTARU', 'NP3RTALU', 'NP3RTARL',
    'NP3RTALL', 'NP3RTALJ', 'NP3RTCON'
]

df_updrs3['UPDRS3_TOTAL'] = df_updrs3[UPDRS3_ITEMS].sum(axis=1, min_count=30)

df_updrs3 = df_updrs3[JOIN_KEY + ['UPDRS3_TOTAL', 'NHY']].dropna(subset=['UPDRS3_TOTAL'])
df_updrs3 = df_updrs3.rename(columns={'NHY': 'HOEHN_YAHR'})

print(f"UPDRS-III loaded: {df_updrs3.shape[0]} rows, {df_updrs3[PATIENT_COL].nunique()} patients")
print(f"Score range:      {df_updrs3['UPDRS3_TOTAL'].min():.0f} – {df_updrs3['UPDRS3_TOTAL'].max():.0f}")
print(f"Mean ± std:       {df_updrs3['UPDRS3_TOTAL'].mean():.1f} ± {df_updrs3['UPDRS3_TOTAL'].std():.1f}")
print(f"\nHoehn & Yahr distribution:")
print(df_updrs3['HOEHN_YAHR'].value_counts().sort_index().to_string())

UPDRS-III loaded: 15817 rows, 2134 patients
Score range:      0 – 100
Mean ± std:       17.5 ± 14.6

Hoehn & Yahr distribution:
HOEHN_YAHR
0.0    4802
1.0    2691
2.0    7456
3.0     732
4.0      99
5.0      31


### 3.4 MDS-UPDRS Part IV - Motor complications

In [36]:
df_updrs4 = pd.read_csv(UPDRS_IV_FILE)

UPDRS4_ITEMS = ['NP4WDYSK', 'NP4DYSKI', 'NP4OFF', 'NP4FLCTI', 'NP4FLCTX', 'NP4DYSTN']

df_updrs4['UPDRS4_TOTAL'] = df_updrs4[UPDRS4_ITEMS].sum(axis=1, min_count=5)
df_updrs4 = df_updrs4[JOIN_KEY + ['UPDRS4_TOTAL']].dropna(subset=['UPDRS4_TOTAL'])

print(f"UPDRS-IV loaded: {df_updrs4.shape[0]} rows, {df_updrs4[PATIENT_COL].nunique()} patients")
print(f"Score range:     {df_updrs4['UPDRS4_TOTAL'].min():.0f} – {df_updrs4['UPDRS4_TOTAL'].max():.0f}")
print(f"Mean ± std:      {df_updrs4['UPDRS4_TOTAL'].mean():.1f} ± {df_updrs4['UPDRS4_TOTAL'].std():.1f}")

UPDRS-IV loaded: 5696 rows, 916 patients
Score range:     0 – 17
Mean ± std:      1.8 ± 2.8


### 3.5 Montreal Cognitive Assessment (MoCA)

In [38]:
df_moca = pd.read_csv(MOCA_FILE)

# MCATOT is the total score (0-30, higher = better cognition)
df_moca = df_moca[JOIN_KEY + ['MCATOT']].dropna(subset=['MCATOT'])
df_moca = df_moca.rename(columns={'MCATOT': 'MOCA_TOTAL'})

print(f"MoCA loaded:  {df_moca.shape[0]} rows, {df_moca[PATIENT_COL].nunique()} patients")
print(f"Score range:  {df_moca['MOCA_TOTAL'].min():.0f} – {df_moca['MOCA_TOTAL'].max():.0f}")
print(f"Mean ± std:   {df_moca['MOCA_TOTAL'].mean():.1f} ± {df_moca['MOCA_TOTAL'].std():.1f}")

MoCA loaded:  7854 rows, 2177 patients
Score range:  0 – 30
Mean ± std:   26.6 ± 3.1


### 3.6 PD Features - Disease duration

In [40]:
df_pdfeat = pd.read_csv(PD_FEAT_FILE)

# PDDXDT = diagnosis date (MM/YYYY), INFODT = visit date
# Compute disease duration in years at time of visit
df_pdfeat['PDDXDT']  = pd.to_datetime(df_pdfeat['PDDXDT'],  format='%m/%Y', errors='coerce')
df_pdfeat['INFODT']  = pd.to_datetime(df_pdfeat['INFODT'],  format='%m/%Y', errors='coerce')
df_pdfeat['DISEASE_DURATION_YRS'] = (
    (df_pdfeat['INFODT'] - df_pdfeat['PDDXDT']).dt.days / 365.25
)

# Keep dominant side of symptoms
df_pdfeat = df_pdfeat[JOIN_KEY + ['DISEASE_DURATION_YRS', 'DOMSIDE']].dropna(subset=['DISEASE_DURATION_YRS'])
df_pdfeat = df_pdfeat[df_pdfeat['DISEASE_DURATION_YRS'] >= 0]  # remove negative durations

print(f"PD Features loaded:   {df_pdfeat.shape[0]} rows, {df_pdfeat[PATIENT_COL].nunique()} patients")
print(f"Duration range:       {df_pdfeat['DISEASE_DURATION_YRS'].min():.1f} – {df_pdfeat['DISEASE_DURATION_YRS'].max():.1f} years")
print(f"Mean ± std:           {df_pdfeat['DISEASE_DURATION_YRS'].mean():.1f} ± {df_pdfeat['DISEASE_DURATION_YRS'].std():.1f} years")

PD Features loaded:   1063 rows, 1063 patients
Duration range:       0.0 – 29.5 years
Mean ± std:           2.9 ± 4.6 years


### 3.7 REM Sleep Disorder


In [41]:
df_rem = pd.read_csv(REM_FILE)

# REM sleep behaviour disorder items (yes/no questions, scored 0/1)
REM_ITEMS = ['DRMVIVID', 'DRMAGRAC', 'DRMNOCTB', 'SLPLMBMV', 'SLPINJUR',
             'DRMVERBL', 'DRMFIGHT', 'DRMUMV', 'DRMOBJFL', 'MVAWAKEN']

df_rem['RBD_SCORE'] = df_rem[REM_ITEMS].sum(axis=1, min_count=8)
df_rem = df_rem[JOIN_KEY + ['RBD_SCORE']].dropna(subset=['RBD_SCORE'])

print(f"REM sleep loaded: {df_rem.shape[0]} rows, {df_rem[PATIENT_COL].nunique()} patients")
print(f"Score range:      {df_rem['RBD_SCORE'].min():.0f} – {df_rem['RBD_SCORE'].max():.0f}")
print(f"Mean ± std:       {df_rem['RBD_SCORE'].mean():.1f} ± {df_rem['RBD_SCORE'].std():.1f}")

REM sleep loaded: 8833 rows, 1929 patients
Score range:      0 – 10
Mean ± std:       2.6 ± 2.5


## 4. Merge Clinical Tables with Latent Vectors

In [156]:
df = df_latents.copy()
n_start = len(df)

clinical_tables = {
    'UPDRS-I':           df_updrs1,
    'UPDRS-II':          df_updrs2,
    'UPDRS-III':         df_updrs3,
    'UPDRS-IV':          df_updrs4,
    'MoCA':              df_moca,
    'PD Features':       df_pdfeat,
    'REM Sleep':         df_rem,
}

for name, df_clin in clinical_tables.items():
    before = len(df)
    df = df.merge(df_clin, on=JOIN_KEY, how='left')
    # display(f"{name}", df.columns.tolist())
    print(f"  Merged {name:<15} → {len(df)} rows (was {before})")

print(f"\nFinal shape: {df.shape}")
print(f"Rows retained: {len(df)}/{n_start} ({len(df)/n_start*100:.1f}%)")
# display(df.columns.tolist())

  Merged UPDRS-I         → 2373 rows (was 2373)
  Merged UPDRS-II        → 2373 rows (was 2373)
  Merged UPDRS-III       → 2766 rows (was 2373)
  Merged UPDRS-IV        → 2766 rows (was 2766)
  Merged MoCA            → 2766 rows (was 2766)
  Merged PD Features     → 2766 rows (was 2766)
  Merged REM Sleep       → 2766 rows (was 2766)

Final shape: (2766, 312)
Rows retained: 2766/2373 (116.6%)


## 5. Coverage Report

In [155]:
CLINICAL_COLS = [
    'UPDRS1_TOTAL', 'UPDRS2_TOTAL', 'UPDRS3_TOTAL', 'UPDRS4_TOTAL',
    'HOEHN_YAHR', 'MOCA_TOTAL', 'DISEASE_DURATION_YRS', 'DOMSIDE', 'RBD_SCORE'
]

print("Clinical variable coverage")
print(f"{'Variable':<25} {'Non-null':>10} {'Coverage':>10}")
print("-" * 47)

for col in CLINICAL_COLS:
    if col in df.columns:
        n     = df[col].notna().sum()
        pct   = n / len(df) * 100
        print(f"  {col:<23} {n:>10} {pct:>9.1f}%")

print(f"\n{'Column':<25} {'PD':>8} {'Control':>10} {'SWEDD':>8}")
print("-" * 53)

for col in ['UPDRS3_TOTAL', 'MOCA_TOTAL', 'DISEASE_DURATION_YRS']:
    if col in df.columns:
        row = f"  {col:<23}"
        # print("m",row)  
        for grp in ['PD', 'Control', 'SWEDD']:
            vals = df[df[LABEL_COL] == grp][col].dropna()
            row += f"  {vals.mean():>8.1f}"
        print(row)

Clinical variable coverage
Variable                    Non-null   Coverage
-----------------------------------------------
  UPDRS1_TOTAL                  1836      66.4%
  UPDRS2_TOTAL                  1836      66.4%
  UPDRS3_TOTAL                  1835      66.3%
  UPDRS4_TOTAL                  1089      39.4%
  HOEHN_YAHR                    1835      66.3%
  MOCA_TOTAL                    2014      72.8%
  DISEASE_DURATION_YRS           554      20.0%
  DOMSIDE                        554      20.0%
  RBD_SCORE                     1321      47.8%

Column                          PD    Control    SWEDD
-----------------------------------------------------
  UPDRS3_TOTAL                 23.3       1.0      15.2
  MOCA_TOTAL                   26.5      28.0      26.3
  DISEASE_DURATION_YRS          1.1       nan       0.4


## 6. Patient-Stratified Train/Val Split
**Why this matters:** Mahmoud's split was row-level — 317 patients appeared in both train and val.  
This version ensures all visits of one patient stay in the same split.

In [172]:
np.random.seed(RANDOM_STATE)


# Get one row per patient with their label
patients = df.drop_duplicates(PATIENT_COL)[[PATIENT_COL, LABEL_COL]]
# print(f"Unique patients: {len(patients)}")
# print(patients.head())
# print("-" * 40)

# Stratify by diagnosis so PD/HC/SWEDD ratio is preserved in both splits
train_patients = set()
for label_val in patients[LABEL_COL].unique():
    group = patients[patients[LABEL_COL] == label_val][PATIENT_COL].tolist()
    # print(f"Label {label_val}: {len(group)} patients")
    np.random.shuffle(group)
    n_train = int(TRAIN_RATIO * len(group))
    train_patients.update(group[:n_train])

df_train = df[df[PATIENT_COL].isin(train_patients)].copy()
df_val   = df[~df[PATIENT_COL].isin(train_patients)].copy() 

# Verify no leakage
overlap = set(df_train[PATIENT_COL]) & set(df_val[PATIENT_COL])
assert len(overlap) == 0, f"DATA LEAKAGE: {len(overlap)} patients in both splits!"

print(f"\nTraining set:")
print(f"  Rows:            {len(df_train)}")
print(f"  Unique patients: {df_train[PATIENT_COL].nunique()}")
print(f"  Labels:          {df_train[LABEL_COL].value_counts().to_dict()}")

print(f"\nValidation set:")
print(f"  Rows:            {len(df_val)}")
print(f"  Unique patients: {df_val[PATIENT_COL].nunique()}")
print(f"  Labels:          {df_val[LABEL_COL].value_counts().to_dict()}")

print(f"\nPatient overlap: {len(overlap)}")




Training set:
  Rows:            2221
  Unique patients: 1148
  Labels:          {'PD': 1948, 'Control': 185, 'SWEDD': 88}

Validation set:
  Rows:            545
  Unique patients: 289
  Labels:          {'PD': 471, 'Control': 48, 'SWEDD': 26}

Patient overlap: 0


## 7. SBR PCA — Fit Once on Training Set
**Why this matters:** Mahmoud refitted the PCA independently in every notebook,  
producing slightly different SBR_PC1 values each time.  
Here we fit once, save the objects, and all downstream notebooks load them.


In [175]:
# Drop rows missing any SBR column
df_train = df_train.dropna(subset=SBR_COLS).copy()
df_val   = df_val.dropna(subset=SBR_COLS).copy()

# Fit on training set only
scaler_sbr = StandardScaler()
sbr_train_scaled = scaler_sbr.fit_transform(df_train[SBR_COLS])

pca_sbr = PCA(n_components=N_SBR_PCS, random_state=RANDOM_STATE)
sbr_train_pcs = pca_sbr.fit_transform(sbr_train_scaled)

for i in range(N_SBR_PCS):
    df_train[f'SBR_PC{i+1}'] = sbr_train_pcs[:, i]

# Apply (never refit) to validation
sbr_val_scaled = scaler_sbr.transform(df_val[SBR_COLS])
sbr_val_pcs    = pca_sbr.transform(sbr_val_scaled)
for i in range(N_SBR_PCS):
    df_val[f'SBR_PC{i+1}'] = sbr_val_pcs[:, i]

# Save fitted objects
with open(SCALER_OUT, 'wb') as f: pickle.dump(scaler_sbr, f)
with open(PCA_OUT,    'wb') as f: pickle.dump(pca_sbr,    f)


print("SBR PCA (fitted on training set only)")
print(f"Explained variance per PC: {pca_sbr.explained_variance_ratio_.round(3)}")
print(f"Total explained:           {pca_sbr.explained_variance_ratio_.sum()*100:.1f}%")
print(f"\nTrain SBR_PC1: mean={df_train['SBR_PC1'].mean():.3f}, std={df_train['SBR_PC1'].std():.3f}")
print(f"Val   SBR_PC1: mean={df_val['SBR_PC1'].mean():.3f},   std={df_val['SBR_PC1'].std():.3f}")
print(f"\nSaved: {SCALER_OUT}")
print(f"Saved: {PCA_OUT}")



SBR PCA (fitted on training set only)
Explained variance per PC: [0.882 0.053 0.049]
Total explained:           98.4%

Train SBR_PC1: mean=-0.000, std=2.301
Val   SBR_PC1: mean=0.073,   std=2.374

Saved: ../../results/models/scaler_sbr.pkl
Saved: ../../results/models/pca_sbr.pkl


## 8. Save Final Files

In [176]:
df_train.to_csv(TRAIN_OUT, index=False)
df_val.to_csv(VAL_OUT,     index=False)

print("Final files saved")
print(f"\nTrain: {TRAIN_OUT}")
print(f"  Shape:           {df_train.shape}")
print(f"  Unique patients: {df_train[PATIENT_COL].nunique()}")
print(f"  Labels:          {df_train[LABEL_COL].value_counts().to_dict()}")

print(f"\nVal:   {VAL_OUT}")
print(f"  Shape:           {df_val.shape}")
print(f"  Unique patients: {df_val[PATIENT_COL].nunique()}")
print(f"  Labels:          {df_val[LABEL_COL].value_counts().to_dict()}")

print(f"\nClinical columns available for correlation analysis:")
clinical_available = [c for c in CLINICAL_COLS if c in df_train.columns
                      and df_train[c].notna().sum() > 100]
for c in clinical_available:
    cov = df_train[c].notna().mean() * 100
    print(f"  {c:<25} {cov:.1f}% coverage in train")

Final files saved

Train: ../../data/processed/clinical_merged/train.csv
  Shape:           (2221, 318)
  Unique patients: 1148
  Labels:          {'PD': 1948, 'Control': 185, 'SWEDD': 88}

Val:   ../../data/processed/clinical_merged/val.csv
  Shape:           (545, 315)
  Unique patients: 289
  Labels:          {'PD': 471, 'Control': 48, 'SWEDD': 26}

Clinical columns available for correlation analysis:
  UPDRS1_TOTAL              67.4% coverage in train
  UPDRS2_TOTAL              67.4% coverage in train
  UPDRS3_TOTAL              67.3% coverage in train
  UPDRS4_TOTAL              39.9% coverage in train
  HOEHN_YAHR                67.3% coverage in train
  MOCA_TOTAL                73.8% coverage in train
  DISEASE_DURATION_YRS      20.3% coverage in train
  DOMSIDE                   20.3% coverage in train
  RBD_SCORE                 48.5% coverage in train
